# Supporting Material Demo Notebook

### Introduction

This notebook is supporting material on **operationalizing large language models**. In this tutorial we will take a small instruction-tuned language model through the toolbox that makes modern LLMs practical: talking to them, training them in less memory, computing attention efficiently, aligning them with rewards and serving them fast.

Sections serve as **introduction**: enough to see it working, measure what it can provide us, and leave optional exercises to dig deeper.

Using the same validation set lets us build a **scoreboard**, so at the end we can compare prompting, fine-tuning and reinforcement learning side by side.

#### Flow of the notebook

The notebook will be divided into separate sections to provide an organized walk through. The sections are:

1. [Importing Python Libraries and preparing the environment](#section01)
2. [Building the domain data and the evaluation harness](#section02)
3. [A practical manual to LLM interaction: Prompting](#section03)
4. [32 bits is wasteful: Automatic Mixed Precision](#section04)
5. [Tuning large models in small spaces: QLoRA](#section05)
6. [Attention faster than O(N^2)?: Flash Attention 2](#section06)
7. [Reinforcement Learning for Language: the TRL library](#section07)
8. [Generating language fast: the vLLM library](#section08)
9. [Closing remarks and scoreboard](#section09)

#### Technical Details

 - Data:
	 - A synthetic **date normalization** task generated inside the notebook.
	 - Each example is a sentence containing a date written in one of eight styles; the target is the same date in ISO 8601 (`YYYY-MM-DD`).
	 - Correctness can be checked by a program, which lets us compare prompting, fine-tuning and reinforcement learning on equal terms.

 - Language Model Used:
	 - [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), a small decoder-only chat model.
	 - [Technical Report](https://arxiv.org/abs/2412.15115)
	 - [Documentation for python](https://huggingface.co/docs/transformers/model_doc/qwen2)

 - Hardware Requirements:
	 - Python 3.10 and above
	 - PyTorch, Transformers, PEFT, bitsandbytes, TRL and vLLM (installed below)
	 - GPU enabled setup. Recommended: compute capability ≥ 8.0 (Colab **L4** or **A100**), where everything runs in bfloat16 and Flash Attention 2 is available.
	 - A Colab **T4** also works: the notebook switches to float16 with loss scaling and skips the Flash Attention 2 cells with a message.

 - Script Objective:
	 - Get the model to normalize dates, first by prompting it, then by fine-tuning and reinforcing it, while measuring the memory and time each technique costs.

#### How to run this notebook

 - Sections 1 and 2 are **always required**: they install the libraries and define the data, the helpers and the scoreboard.
 - Sections 3 to 7 only depend on sections 1 and 2, so you can jump straight to any of them.
 - Section 8 serves the adapters saved by sections 5 and 7 if they exist on disk, and skips them otherwise.
 - After a runtime restart, re-run sections 1 and 2 first. Files under `adapters/` survive a restart (but not a disconnection).
 - Rough running times on an L4 (expect about twice as long on a T4): installation 3-5 min, prompting 2 min, mixed precision 3 min, QLoRA 4 min, Flash Attention 2 min, TRL 8 min, vLLM 4 min.

---
***NOTE***
- *Every technique here was designed for models with billions of parameters. We use a 0.5B model so that each section runs in minutes, a manageable time window; the ideas transfer unchanged to larger models.*

> **About libraries**: *Library versions are pinned to the ones this notebook was written against. These libraries change fast; if you remove the pins, expect errors.*
---

<a id='section01'></a>
### Importing Python Libraries and preparing the environment

At this step we will be importing the libraries and modules needed to run our script. Libraries are:
* Pandas and Matplotlib
* PyTorch
* Transformers and Datasets (from huggingface)
* PEFT and bitsandbytes for QLoRA
* TRL for reinforcement learning
* vLLM for fast generation
* kernels, to download pre-compiled Flash Attention kernels instead of compiling them

Followed by that we will prepare the device for GPU execution and check what our GPU supports.

---
***NOTE***
- *vLLM requires an exact PyTorch version, so the installation replaces the one that comes with Colab. Run it **before** importing torch. If Colab asks you to restart the session afterwards, do so and continue from the imports cell.*
---

In [ ]:
# Installing the transformers library and additional libraries (3-5 minutes)

!uv pip install -q vllm==0.31.0 trl==1.14.2 peft==0.21.2 bitsandbytes==0.50.2 kernels==0.17.2 torchao==0.16.0 datasets accelerate

In [ ]:
# Importing stock ml libraries
import gc
import os
import re
import json
import math
import time
import random
import datetime
import itertools
import contextlib
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset as HFDataset

print(f"torch {torch.__version__} | transformers {transformers.__version__}")

In [ ]:
# # Setting up the device for GPU usage

from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

In [ ]:
# Checking what the GPU supports: bfloat16 and Flash Attention 2 need compute capability >= 8.0
assert device == 'cuda', "This notebook needs a GPU runtime (Runtime > Change runtime type)"

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
IS_AMPERE_OR_NEWER = CAPABILITY >= (8, 0)
AMP_DTYPE = torch.bfloat16 if IS_AMPERE_OR_NEWER else torch.float16

print(f"GPU: {GPU_NAME} | compute capability {CAPABILITY} | {GPU_MEMORY_GB:.1f} GB")
print(f"Half precision type for this notebook: {AMP_DTYPE}")
if not IS_AMPERE_OR_NEWER:
    print("Pre-Ampere GPU: float16 + loss scaling will be used and Flash Attention 2 cells will be skipped.")

In [ ]:
# @title Customize your key variables here
# Sections of config

# Defining some key variables that will be used later on
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct" # @param {type:"string"}
SEED = 42 # @param {type:"integer"}
N_TRAIN = 2000 # @param {type:"integer"}
N_VALID = 300 # @param {type:"integer"}
N_EVAL = 300 # @param {type:"integer"}
MAX_NEW_TOKENS = 16 # @param {type:"integer"}
TRAIN_BATCH_SIZE = 4 # @param {type:"integer"}
EVAL_BATCH_SIZE = 32 # @param {type:"integer"}
LEARNING_RATE = 1e-5 # @param {type:"number"}

random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Small utilities to measure time and memory, and to free the GPU between sections

def release(*names):
    # Deletes global variables by name and gives their GPU memory back
    for name in names:
        globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()


def free_gpu_gb():
    free, total = torch.cuda.mem_get_info()
    return free / 1024**3


class Measure:
    # Context manager that records wall-clock seconds and peak GPU memory (GB) of a block
    def __enter__(self):
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        self.start = time.perf_counter()
        return self

    def __exit__(self, *exc):
        torch.cuda.synchronize()
        self.seconds = time.perf_counter() - self.start
        self.peak_gb = torch.cuda.max_memory_allocated() / 1024**3

<a id='section02'></a>
### Building the domain data and the evaluation harness

We will generate our own data. A random date is written in one of eight styles and dropped into a sentence. The target is always the ISO 8601 date.

---
***NOTE***
- *Dotted dates such as `5.3.2021` follow the day-first convention (5 March). A model can only know it from the prompt or from training data. Keep an eye on the `dotted` style in the error analyses.*
---

Following this we prepare the pieces that every section reuses:
- `build_messages` turns a sentence into a chat conversation for the model.
- `score` compares a prediction with the answer in two ways:
	- **strict**: the output is exactly the date and nothing else.
	- **lenient**: the correct date appears somewhere in the output.
- `generate` and `evaluate` run a model over the validation split and report both accuracies with a 95% confidence interval. With a few hundred examples, differences of a few points are noise!
- `majority_vote` picks the most common date among several sampled answers (for self-consistency).
- `SCOREBOARD` collects every named evaluation, and `COSTS` every time and memory measurement, for the final comparison.
- `DateSFTDataset`, `DataCollator` and `train_step` are shared by the training sections.

In [ ]:
# Generating sentences that contain a date, together with their ISO 8601 normalization

MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]


def ordinal(n):
    return "th" if 11 <= n % 100 <= 13 else {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")


DATE_STYLES = {
    "day month year": lambda d: f"{d.day} {MONTHS[d.month - 1]} {d.year}",
    "month day, year": lambda d: f"{MONTHS[d.month - 1]} {d.day}, {d.year}",
    "abbreviated": lambda d: f"{MONTHS[d.month - 1][:3]} {d.day} {d.year}",
    "ordinal": lambda d: f"the {d.day}{ordinal(d.day)} of {MONTHS[d.month - 1]}, {d.year}",
    "slashed": lambda d: f"{d.year}/{d.month:02d}/{d.day:02d}",
    "weekday": lambda d: f"{WEEKDAYS[d.weekday()]}, {MONTHS[d.month - 1]} {d.day}, {d.year}",
    "military": lambda d: f"{d.day:02d}-{MONTHS[d.month - 1][:3].upper()}-{d.year}",
    "dotted": lambda d: f"{d.day}.{d.month}.{d.year}",
}

CONTEXTS = [
    "The invoice was issued on {date}.",
    "Please submit the report before {date}.",
    "She was born on {date} in a small coastal town.",
    "Deadline: {date} (no extensions).",
    "We met for the first time on {date}, if I remember correctly.",
    "Shipment expected {date}",
    "According to the minutes, the vote took place on {date}.",
    "Opening day is {date}!",
]


def make_example(rng):
    first = datetime.date(1950, 1, 1).toordinal()
    last = datetime.date(2035, 12, 31).toordinal()
    date = datetime.date.fromordinal(rng.randint(first, last))
    style = rng.choice(list(DATE_STYLES))
    text = rng.choice(CONTEXTS).format(date=DATE_STYLES[style](date))
    return {"text": text, "answer": date.isoformat(), "style": style}


def build_split(n, seed):
    rng = random.Random(seed)
    return [make_example(rng) for _ in range(n)]


train_data = build_split(N_TRAIN, SEED)
valid_data = build_split(N_VALID, SEED + 1)

print("TRAIN Dataset: {}".format(len(train_data)))
print("VALID Dataset: {}".format(len(valid_data)))
pd.DataFrame(train_data[:8])

In [ ]:
# Turning a sentence into a conversation and scoring the answers

SYSTEM_PROMPT = "You normalize dates. Reply only with the date in ISO 8601 format (YYYY-MM-DD)."
ISO_PATTERN = re.compile(r"\d{4}-\d{2}-\d{2}")


def build_messages(text, system_prompt=SYSTEM_PROMPT, shots=()):
    messages = [{"role": "system", "content": system_prompt}]
    for shot in shots:
        messages.append({"role": "user", "content": shot["text"]})
        messages.append({"role": "assistant", "content": shot["answer"]})
    messages.append({"role": "user", "content": text})
    return messages


def score(prediction, answer):
    prediction = prediction.strip()
    found = ISO_PATTERN.search(prediction)
    return {
        "strict": float(prediction == answer),
        "lenient": float(found is not None and found.group(0) == answer),
    }

In [ ]:
# Accuracy with a 95% Wilson confidence interval, plus time and memory costs

SCOREBOARD = []
COSTS = []


def wilson_interval(p, n, z=1.96):
    center = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return max(0.0, center - half), min(1.0, center + half)


def report(results, name=None):
    n = len(results)
    line = []
    row = {"approach": name, "n": n}
    for metric in ["strict", "lenient"]:
        p = results[metric].mean()
        low, high = wilson_interval(p, n)
        row.update({metric: p, f"{metric}_low": low, f"{metric}_high": high})
        line.append(f"{metric.capitalize()} accuracy = {p:.3f} [{low:.2f}, {high:.2f}]")
    print(" | ".join(line))
    if name is not None:
        SCOREBOARD[:] = [r for r in SCOREBOARD if r["approach"] != name]  # re-running a cell replaces its row
        SCOREBOARD.append(row)
    return results


def record_cost(name, **values):
    COSTS[:] = [r for r in COSTS if r["what"] != name]
    COSTS.append({"what": name, **values})

In [ ]:
# Loading the tokenizer and defining generation and evaluation
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"  # decoder-only models append new tokens on the right, so padding goes left

GREEDY = dict(do_sample=False, temperature=None, top_p=None, top_k=None) # A greedy decoding strategy!


@torch.no_grad()
def generate(model, batch_messages, max_new_tokens=MAX_NEW_TOKENS, **generation_kwargs):
    texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch_messages]
    inputs = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id,
                             **{**GREEDY, **generation_kwargs})
    return tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def evaluate(model, data, name=None, n=N_EVAL, batch_size=EVAL_BATCH_SIZE, build=build_messages,
             parse=lambda prediction: prediction, **generation_kwargs):
    model.eval()
    n = min(n, len(data))
    rows = []
    for start in range(0, n, batch_size):
        batch = data[start:min(start + batch_size, n)]
        predictions = generate(model, [build(x["text"]) for x in batch], **generation_kwargs)
        for example, prediction in zip(batch, predictions):
            rows.append({**example, "prediction": prediction, **score(parse(prediction), example["answer"])})
    return report(pd.DataFrame(rows), name)


def accuracy_by_style(results):
    return results.groupby("style")[["strict", "lenient"]].mean().sort_values("strict")


def majority_vote(predictions):
    # Most common ISO date among several predictions ("" if none contains a date)
    dates = [m.group(0) for m in map(ISO_PATTERN.search, predictions) if m]
    return Counter(dates).most_common(1)[0][0] if dates else ""

In [ ]:
# Creating the dataset, the dataloader and the training step for the training sections

class DateSFTDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        example = self.data[index]
        messages = build_messages(example["text"])
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        full = self.tokenizer.apply_chat_template(
            messages + [{"role": "assistant", "content": example["answer"]}], tokenize=False)
        assert full.startswith(prompt)
        prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
        completion_ids = self.tokenizer(full[len(prompt):], add_special_tokens=False)["input_ids"]
        # Only the answer is learned: prompt positions get the ignore index -100
        return {
            "input_ids": prompt_ids + completion_ids,
            "labels": [-100] * len(prompt_ids) + completion_ids,
        }


class DataCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, input_batch):
        max_len = max(len(x["input_ids"]) for x in input_batch)
        input_ids, attention_mask, labels = [], [], []
        for x in input_batch:
            pad = max_len - len(x["input_ids"])
            input_ids.append(x["input_ids"] + [self.pad_token_id] * pad)
            attention_mask.append([1] * len(x["input_ids"]) + [0] * pad)
            labels.append(x["labels"] + [-100] * pad)
        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention_mask),
            "labels": torch.tensor(labels),
        }


def train_step(model, data, optimizer, scaler, amp_dtype):
    # One optimization step. amp_dtype=None trains in float32; otherwise autocast + (for float16) loss scaling
    optimizer.zero_grad(set_to_none=True)
    data = {key: value.to(device) for key, value in data.items()}
    autocast = torch.autocast(device_type="cuda", dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()
    with autocast:
        loss = model(**data).loss
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    return loss.item()


training_set = DateSFTDataset(train_data, tokenizer)

train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0,
                'collate_fn': DataCollator(tokenizer.pad_token_id)
                }

training_loader = DataLoader(training_set, **train_params)
print("Tokens of one training example that receive a loss (the prompt is masked out):")
sample = training_set[0]
print(repr(tokenizer.decode([t for t, l in zip(sample["input_ids"], sample["labels"]) if l != -100])))

<a id='section03'></a>
### A practical manual to LLM interaction: Prompting

Before training anything we should find out how far we get by just *asking*. A few facts that explain most prompting bugs:
 - A chat model does not see "messages": it sees **one string** where each turn is wrapped in special tokens. The **chat template** stored in the tokenizer builds the string for the model to see.
 - The **system** turn sets standing instructions, **user** turns carry the input, **assistant** turns are the model's own (or our pretended) answers.
 - `add_generation_prompt=True` appends the header of an empty assistant turn. Without it the model may continue the *user's* text instead of answering.
 - Batched generation pads on the **left**, so that every sequence ends exactly where generation starts.
 - Decoding is a choice: **greedy** takes the most likely token; **sampling** draws from the distribution, sharpened or flattened by the `temperature`. Different decoding generates different texts.

In this section we measure the **zero-shot** baseline, add **few-shot** demonstrations, and look at **sampling**.

#### Further Reading
- [Chat templates in transformers](https://huggingface.co/docs/transformers/chat_templating)
- [Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165)
- [Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903)
- [Self-Consistency Improves Chain of Thought Reasoning](https://arxiv.org/abs/2203.11171)

In [ ]:
# Looking under the hood: the chat template turns a list of messages into one string
messages = build_messages(valid_data[0]["text"])
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
print(f"{len(tokenizer(prompt)['input_ids'])} tokens")

without_header = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print("add_generation_prompt=True appends:", repr(prompt[len(without_header):]))

In [ ]:
# Loading the model in half precision for inference
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=AMP_DTYPE, device_map=device)
print(f"Model memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

#### Zero-shot
Only the system instruction and the sentence. Look at the failures and at the per-style table: are they wrong dates, or right dates in the wrong format?

In [ ]:
zero_shot_results = evaluate(model, valid_data, name="Zero-shot prompt")
accuracy_by_style(zero_shot_results)

In [ ]:
zero_shot_results[zero_shot_results.strict == 0][["text", "answer", "prediction"]].head(10)

#### Few-shot
We prepend a handful of solved examples as earlier user/assistant turns. The model imitates the format of its own "previous answers".

In [ ]:
FEW_SHOT_EXAMPLES = build_split(4, seed=SEED + 100)


def build_few_shot(text):
    return build_messages(text, shots=FEW_SHOT_EXAMPLES)


few_shot_results = evaluate(model, valid_data, name="Few-shot prompt (4 shots)", build=build_few_shot)
pd.concat({"zero-shot": accuracy_by_style(zero_shot_results).strict,
           "few-shot": accuracy_by_style(few_shot_results).strict}, axis=1)

#### Decoding parameters
`generate` is greedy by default in our helper. With `do_sample=True` we draw tokens from the distribution: a low `temperature` makes it almost greedy, a high one makes it creative... and wrong.

In [ ]:
# Sampling the same prompt several times at different temperatures
example = valid_data[3]
print("Sentence:", example["text"], "| Answer:", example["answer"])
for temperature in [0.1, 0.7, 1.5]:
    samples = generate(model, [build_messages(example["text"])], do_sample=True, temperature=temperature,
                       top_p=1.0, top_k=0, num_return_sequences=5)
    print(f"T={temperature}: {samples}")

#### **First coding exercise**: how many shots?
Measure accuracy as a function of the number of demonstrations:

1.   Build `k` shots with `build_split(k, seed=SEED + 100)` for each `k` in `SHOT_COUNTS`
2.   Create a `build` function that injects them with `build_messages(text, shots=shots)`
3.   Run `evaluate(model, valid_data, build=build)` and append the strict and lenient accuracy to `sweep`
4.   Run the plotting cell. Where does it saturate? Are the gaps larger than the confidence intervals?

In [ ]:
SHOT_COUNTS = [0, 1, 2, 4, 8, 16]
sweep = []
for k in SHOT_COUNTS:
    # Your code goes here!
    shots = None
    build = None
    results = None
    sweep.append({"k": k, "strict": None, "lenient": None})

In [ ]:
# Useful cell: plotting the sweep
sweep_df = pd.DataFrame(sweep)
sweep_df.plot(x="k", y=["strict", "lenient"], marker="o", logx=False, figsize=(6, 3.5))
plt.xlabel("number of shots")
plt.ylabel("accuracy")
plt.ylim(0, 1)
plt.show()

#### **Second coding exercise**: self-consistency
Sample several answers and keep the most voted one. `majority_vote` (defined in section 2) already extracts the dates and counts them.

1.   Sample `n_samples` completions with `generate(model, [messages], do_sample=True, temperature=temperature, num_return_sequences=n_samples)`
2.   Return `majority_vote(samples)`
3.   Run the check cell: does voting beat greedy decoding on the same examples?

In [ ]:
def self_consistency(model, text, n_samples=8, temperature=0.7):
    messages = build_messages(text)
    # Your code goes here!
    return None

In [ ]:
# Useful cell: comparing voting against greedy on the same 60 examples
subset = valid_data[:60]
votes = pd.DataFrame([{**x, "prediction": self_consistency(model, x["text"])} for x in subset])
votes = pd.concat([votes, pd.DataFrame([score(p, a) for p, a in zip(votes.prediction, votes.answer)])], axis=1)
print("Self-consistency:", end=" "); report(votes)
print("Greedy:          ", end=" "); evaluate(model, subset, n=len(subset))

#### **Third coding exercise**: prompt engineering
Prompt engineering is an empirical discipline. Write at least three alternative system prompts and compare them.

1.   Add your prompts to `CANDIDATE_SYSTEM_PROMPTS`
2.   For each one, build the messages with `build_messages(text, system_prompt=...)` and evaluate
3.   Look at the `dotted` row of `accuracy_by_style`: can you fix it by stating the day-first convention in the prompt?

**Change one thing at a time, or you will not know what helped!**

In [ ]:
CANDIDATE_SYSTEM_PROMPTS = [
    SYSTEM_PROMPT,
    # Your prompts go here!
]

prompt_results = {}
for system_prompt in CANDIDATE_SYSTEM_PROMPTS:
    # Your code goes here!
    prompt_results[system_prompt] = None

In [ ]:
# Useful cell: one row per prompt, one column per date style
pd.DataFrame({prompt[:60]: accuracy_by_style(results).strict
              for prompt, results in prompt_results.items() if results is not None}).T

#### **Last coding exercise**: chain-of-thought
Ask the model to reason before answering.

1.   Write `COT_SYSTEM_PROMPT` asking for a short reasoning followed by a last line `ANSWER: YYYY-MM-DD`
2.   Implement `parse_cot` so that it returns only the date after `ANSWER:` (or `""` if missing)
3.   Run the evaluation cell (it allows 128 new tokens). Does reasoning help a 0.5B model? Look at a few outputs before concluding

In [ ]:
COT_SYSTEM_PROMPT = None


def build_cot(text):
    return build_messages(text, system_prompt=COT_SYSTEM_PROMPT)


def parse_cot(prediction):
    # Your code goes here!
    return None

In [ ]:
# Useful cell: evaluating with reasoning (slower: many more tokens per answer)
cot_results = evaluate(model, valid_data, name="Chain-of-thought prompt", n=100,
                       build=build_cot, parse=parse_cot, max_new_tokens=128)
cot_results[["text", "answer", "prediction"]].head(5)

<a id='section04'></a>
### 32 bits is wasteful: Automatic Mixed Precision

Deep learning does not need 32-bit floats everywhere. Lower precision formats use less memory and run much faster on tensor cores, but they trade **range** (exponent bits) or **precision** (mantissa bits):

| Format | Sign | Exponent | Mantissa | Typical issue |
|---|---|---|---|---|
| float32 | 1 | 8 | 23 | none, but slow and big |
| TF32 (Ampere+, matmuls only) | 1 | 8 | 10 | slightly less precise float32 matmuls, for free |
| float16 | 1 | 5 | 10 | small range: overflow and gradient underflow |
| bfloat16 (Ampere+) | 1 | 8 | 7 | low precision: small updates get rounded away |

**Automatic Mixed Precision (AMP)** keeps the **weights in float32** ("master weights") and runs each operation in the cheapest safe precision:
 - `torch.autocast` runs matrix multiplications in half precision and keeps sensitive ops (softmax, norms, losses) in float32.
 - `torch.amp.GradScaler` multiplies the loss by a large factor before `backward()` so that float16 gradients do not underflow to zero, and divides it back before the optimizer step. It skips steps with `inf` gradients and lowers the factor. bfloat16 has the range of float32, so it does not need it.

---
***NOTE***
- *AMP shrinks **activations** and speeds up **matmuls**. It does not shrink weights, gradients or optimizer states, which stay in float32. For a small model with a huge vocabulary like ours, expect a clear speedup but modest memory savings.*
---

#### Further Reading
- [Mixed Precision Training](https://arxiv.org/abs/1710.03740)
- [PyTorch AMP documentation](https://pytorch.org/docs/stable/amp.html)
- [bfloat16 explained](https://cloud.google.com/tpu/docs/bfloat16)

In [ ]:
# Releasing the inference model of the previous section (if any)
release("model")

# Comparing the numerical formats
rows = []
for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    info = torch.finfo(dtype)
    rows.append({"dtype": str(dtype), "bits": info.bits, "max": info.max,
                 "smallest normal": info.tiny, "epsilon": info.eps})
pd.DataFrame(rows)

In [ ]:
# Three ways in which low precision bites
big, small = torch.tensor(70000.0), torch.tensor(1e-8)
print("Overflow : fp16 ->", big.half().item(), "| bf16 ->", big.bfloat16().item())
print("Underflow: fp16 ->", small.half().item(), "| bf16 ->", small.bfloat16().item())
print("Rounding : in bf16, 256 + 1 =", (torch.tensor(256.0, dtype=torch.bfloat16) + 1).item())

In [ ]:
# Measuring matrix multiplication throughput in each precision
def matmul_tflops(dtype, n=4096, repeats=20, fp32_precision="ieee"):
    previous = torch.backends.cuda.matmul.fp32_precision
    torch.backends.cuda.matmul.fp32_precision = fp32_precision  # "tf32" lets float32 matmuls use tensor cores
    a = torch.randn(n, n, device=device, dtype=dtype)
    b = torch.randn(n, n, device=device, dtype=dtype)
    for _ in range(3):  # warm-up
        a @ b
    with Measure() as m:
        for _ in range(repeats):
            a @ b
    torch.backends.cuda.matmul.fp32_precision = previous
    return 2 * n**3 * repeats / m.seconds / 1e12


setups = [("float32", torch.float32, "ieee"), ("float16", torch.float16, "ieee")]
if IS_AMPERE_OR_NEWER:
    setups += [("float32 as TF32", torch.float32, "tf32"), ("bfloat16", torch.bfloat16, "ieee")]
for label, dtype, precision in setups:
    print(f"{label:16s} {matmul_tflops(dtype, fp32_precision=precision):7.1f} TFLOP/s")

#### Mixed precision training
We now fine-tune the **full** model in float32 and with AMP, using the shared `train_step` from section 2: it is the usual step plus an `autocast` context and the scaler.

To compare fairly we run a few **warm-up** steps first (they pay for CUDA initialization and allocate the optimizer states) and then measure:
 - the memory that stays allocated between steps: weights and optimizer states,
 - the extra memory during a step: activations and gradients,
 - the time per step.

In [ ]:
# @title Customize your key variables here
AMP_STEPS = 20 # @param {type:"integer"}
AMP_WARMUP_STEPS = 3 # @param {type:"integer"}
AMP_BATCH_SIZE = 8 # @param {type:"integer"}

In [ ]:
amp_loader = DataLoader(training_set, batch_size=AMP_BATCH_SIZE, shuffle=True,
                        collate_fn=DataCollator(tokenizer.pad_token_id))


def run_precision_experiment(amp_dtype, steps=AMP_STEPS, warmup_steps=AMP_WARMUP_STEPS, cast_model_to=None):
    torch.manual_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,
                                                 dtype=cast_model_to or torch.float32,
                                                 device_map=device)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16)
    batches = itertools.cycle(amp_loader)
    losses = [train_step(model, next(batches), optimizer, scaler, amp_dtype) for _ in range(warmup_steps)]
    optimizer.zero_grad(set_to_none=True)
    resident_gb = torch.cuda.memory_allocated() / 1024**3
    with Measure() as m:
        for _ in range(steps):
            losses.append(train_step(model, next(batches), optimizer, scaler, amp_dtype))
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    label = f"weights {cast_model_to or torch.float32}, autocast {amp_dtype}"
    return {"setup": label, "seconds/step": m.seconds / steps,
            "weights + optimizer (GB)": resident_gb,
            "activations + gradients (GB)": m.peak_gb - resident_gb, "losses": losses}


experiments = [run_precision_experiment(None), run_precision_experiment(AMP_DTYPE)]
for e in experiments:
    record_cost(f"Full fine-tuning, {e['setup']}", **{k: v for k, v in e.items() if k not in ("setup", "losses")})
pd.DataFrame(experiments).drop(columns="losses")

In [ ]:
for experiment in experiments:
    plt.plot(experiment["losses"], label=experiment["setup"])
plt.axvline(AMP_WARMUP_STEPS, color="grey", linestyle=":", label="end of warm-up")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

#### **First coding exercise**: the autocast policy
Autocast does not cast everything. Find out its policy:

1.   Fill each entry of `ops` with a function of `x` using `@`, `torch.softmax`, `torch.nn.functional.layer_norm`, `.sum()` and `torch.exp`
2.   **Predict** the output dtype of each op before running the useful cell
3.   Run it and explain the pattern: which ops are "safe" in half precision and which are not?

In [ ]:
ops = {
    "matmul": lambda x: None,
    "softmax": lambda x: None,
    "layer_norm": lambda x: None,
    "sum": lambda x: None,
    "exp": lambda x: None,
}
# Your code goes here!

In [ ]:
# Useful cell: output dtypes under autocast
x = torch.randn(64, 64, device=device)
with torch.autocast("cuda", dtype=AMP_DTYPE):
    for name, fn in ops.items():
        out = fn(x)
        print(f"{name:12s} -> {out.dtype if out is not None else 'not implemented'}")

#### **Second coding exercise**: loss scaling by hand
In real AMP the master weight is float32, the computation is float16, and the gradient is unscaled **in float32**. Reproduce it with a tiny graph whose true gradient is `1e-8`:

1.   Create `w = torch.ones(1, device=device, requires_grad=True)` (float32) and `x = torch.full((1,), 1e-3, device=device, dtype=torch.float16)`
2.   Compute `y = (w.half() * x) * torch.tensor(1e-5, device=device, dtype=torch.float16)`: the gradient of `y` with respect to `w` is `1e-8`
3.   Backpropagate `y.sum().float() * scale` and return `w.grad.item() / scale`
4.   Run the useful cell: which scales recover `1e-8`? What happens below and above that window, and why does `GradScaler` start at `2^16` and *halve* the scale when it sees `inf`?

In [ ]:
def scaled_gradient(scale):
    # Your code goes here!
    return None

In [ ]:
# Useful cell: sweeping the loss scale
for power in [0, 4, 8, 12, 15, 16, 20]:
    print(f"scale = 2^{power:<2d} -> recovered gradient {scaled_gradient(2.0**power)}")
print("GradScaler initial scale:", torch.amp.GradScaler("cuda").get_scale())

#### **Third coding exercise**: why keep float32 master weights?
An Adam step moves each weight by roughly `lr` in absolute value. If that is smaller than the spacing between representable numbers around the weight, the update is **rounded away**.

1.   Implement `fraction_of_lost_updates`: cast each parameter to `dtype`, add `lr` (also in `dtype`) and count the entries that did not change
2.   Run the useful cell for `LEARNING_RATE` and for `100 * LEARNING_RATE`
3.   Run the pure half-precision training (`cast_model_to=AMP_DTYPE`, no autocast) and compare its loss curve with AMP. Do your numbers explain it?

---
***NOTE***
- *On a T4 (float16) pure half training with Adam may even produce NaN. That is part of the lesson.*
---

In [ ]:
def fraction_of_lost_updates(model, lr, dtype=AMP_DTYPE):
    lost, total = 0, 0
    for w in model.parameters():
        # Your code goes here!
        pass
    return lost / max(total, 1)

In [ ]:
# Useful cell: measuring lost updates, then training in pure half precision
probe = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32, device_map=device)
for lr in [LEARNING_RATE, 100 * LEARNING_RATE]:
    print(f"lr={lr:g}: {fraction_of_lost_updates(probe, lr):.1%} of the updates are rounded away in {AMP_DTYPE}")
release("probe")

pure_half = run_precision_experiment(None, cast_model_to=AMP_DTYPE)
for experiment in experiments + [pure_half]:
    plt.plot(experiment["losses"], label=experiment["setup"])
plt.legend()
plt.show()

#### Last coding exercise: the batch size frontier
Find the largest batch size that fits in memory in float32 and with AMP:

1.   In `fits`, build a batch of `batch_size` copies of a training example with `DataCollator`, load a fresh float32 model and run **two** `train_step`s (the second one allocates Adam's states)
2.   Return `True`, or `False` if `torch.OutOfMemoryError` is raised
3.   Always delete the model and optimizer and call `torch.cuda.empty_cache()` before returning, even after an error (`try` / `finally`)

In [ ]:
def fits(batch_size, amp_dtype):
    model, optimizer = None, None
    try:
        # Your code goes here!
        return None
    except torch.OutOfMemoryError:
        return False
    finally:
        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
# Useful cell: doubling the batch until it no longer fits
for amp_dtype in [None, AMP_DTYPE]:
    batch_size = 32
    while batch_size <= 4096 and fits(batch_size, amp_dtype):
        batch_size *= 2
    print(f"autocast {amp_dtype}: largest power-of-two batch that fits = {batch_size // 2}")

<a id='section05'></a>
### Tuning large models in small spaces: QLoRA

Full fine-tuning with Adam needs about 16 bytes per parameter (float32 weights, gradients and two optimizer moments). For a 7B model that is more than 100 GB. QLoRA brings it down to a single consumer GPU by combining two ideas:

#### LoRA
 - The pretrained weight `W` is **frozen**. We learn a low-rank update `ΔW = B·A` with `A ∈ R^{r×k}`, `B ∈ R^{d×r}` and `r ≪ min(d, k)`.
 - The layer computes `W·x + (α/r)·B·A·x`. `B` starts at zero, so training starts exactly from the pretrained model.
 - Only `A` and `B` receive gradients and optimizer states: usually around 1% of the parameters.

#### Quantization
 - The frozen weights are stored in **4-bit NormalFloat (NF4)**: 16 levels placed at the quantiles of a normal distribution, which is how trained weights are distributed.
 - **Double quantization** also quantizes the per-block scaling constants.
 - Weights are de-quantized on the fly to the `compute_dtype` for each matrix multiplication.
 - **Paged optimizers** move optimizer states to CPU memory when the GPU runs out, instead of crashing on memory spikes.

---
***NOTE***
- *Gradients still flow **through** the frozen 4-bit weights to reach the LoRA matrices of earlier layers; they are just never applied to them.*
- *Only the linear layers are quantized. The embedding matrix (151,936 tokens × 896 dimensions) stays in 16 bits and is about a quarter of this small model, so here the footprint shrinks much less than 4x. For a 7B model the linear layers dominate and the ratio gets close to 4x.*
- *`prepare_model_for_kbit_training` turns on **gradient checkpointing**: activations are recomputed in the backward pass instead of stored. It trades time for memory, so do not compare its speed with the AMP section.*
---

#### Further Reading
- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)
- [PEFT documentation](https://huggingface.co/docs/peft)
- [bitsandbytes documentation](https://huggingface.co/docs/bitsandbytes)

In [ ]:
# @title Customize your key variables here
LORA_R = 16 # @param {type:"integer"}
LORA_ALPHA = 32 # @param {type:"integer"}
LORA_DROPOUT = 0.05 # @param {type:"number"}
QLORA_STEPS = 120 # @param {type:"integer"}
QLORA_LEARNING_RATE = 2e-4 # @param {type:"number"}
QLORA_ADAPTER_DIR = "adapters/qlora-dates" # @param {type:"string"}

In [ ]:
# Baseline: the 16-bit footprint, and how much of it the (never quantized) embeddings take
release("model")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=AMP_DTYPE, device_map=device)
footprint_16bit = model.get_memory_footprint() / 1024**3
n_params = sum(p.numel() for p in model.parameters())
n_embedding = model.get_input_embeddings().weight.numel()
print(f"16-bit footprint: {footprint_16bit:.2f} GB | embeddings are {n_embedding / n_params:.0%} of the parameters")
release("model")

In [ ]:
# Loading the model in 4 bits
import bitsandbytes as bnb

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=AMP_DTYPE,
)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map=device)
footprint_4bit = model.get_memory_footprint() / 1024**3
print(f"4-bit footprint: {footprint_4bit:.2f} GB ({footprint_16bit / footprint_4bit:.1f}x smaller)")

# A quantized layer stores packed uint8 values (two 4-bit numbers per byte) plus a quantization state
layer = model.model.layers[0].self_attn.q_proj
print(type(layer).__name__, layer.weight.dtype, tuple(layer.weight.shape))
release("model")

In [ ]:
# Reusable helpers: build a QLoRA model and train its adapters with a paged 8-bit optimizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


def build_qlora_model(r=LORA_R, alpha=LORA_ALPHA, quant_type="nf4"):
    config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=quant_type,
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=AMP_DTYPE)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=config, device_map=device)
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=LORA_DROPOUT,
                             target_modules="all-linear", task_type="CAUSAL_LM")
    return get_peft_model(model, lora_config)


def print_progress(step, loss, every=20):
    if step % every == 0:
        print(f"Step: {step}, Loss: {loss:.4f}")


def train_adapter(model, steps=QLORA_STEPS, learning_rate=QLORA_LEARNING_RATE, every=20):
    optimizer = bnb.optim.PagedAdamW8bit([p for p in model.parameters() if p.requires_grad], lr=learning_rate)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
    batches = itertools.cycle(training_loader)
    model.train()
    losses = []
    with Measure() as m:
        for step in range(steps):
            losses.append(train_step(model, next(batches), optimizer, scaler, AMP_DTYPE))
            print_progress(step, losses[-1], every)
    return losses, m

In [ ]:
# Attaching the LoRA adapters and scoring the model before training
model = build_qlora_model()
model.print_trainable_parameters()
before_results = evaluate(model, valid_data, name="4-bit base model (before QLoRA)")

In [ ]:
# Training only the adapters
losses, m = train_adapter(model)
record_cost("QLoRA fine-tuning (4-bit base, LoRA adapters)", **{"seconds/step": m.seconds / QLORA_STEPS,
            "peak memory (GB)": m.peak_gb})
print(f"{m.seconds:.0f} s | peak memory {m.peak_gb:.2f} GB")
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("loss")
plt.show()

In [ ]:
# Score after training, and saving the adapter (only the LoRA weights are written: a few MB)
after_results = evaluate(model, valid_data, name="QLoRA fine-tuned")
print(accuracy_by_style(after_results))
model.save_pretrained(QLORA_ADAPTER_DIR)
!ls -lh {QLORA_ADAPTER_DIR}

#### **First coding exercise**: counting LoRA parameters
Count the LoRA parameters by hand:

1.   For a linear layer with input size `k` and output size `d`, LoRA adds `r·(k + d)` parameters
2.   Read the shapes from `config`: `hidden_size`, `intermediate_size`, `num_attention_heads`, `num_key_value_heads` (with `head_dim = hidden_size // num_attention_heads`) and `num_hidden_layers`
3.   Sum over the 7 projections of each layer: `q`, `k`, `v`, `o` (attention) and `gate`, `up`, `down` (MLP). Careful: `k` and `v` are smaller because of grouped-query attention!

In [ ]:
def count_lora_parameters(config, r):
    # Your code goes here!
    return None

In [ ]:
# Useful cell: checking against PEFT
expected = model.get_nb_trainable_parameters()[0]
print(f"yours: {count_lora_parameters(model.config, LORA_R)} | PEFT: {expected}")

#### **Second coding exercise**: rank sweep
Is more rank always better on this task?

1.   For each `r` in `RANKS`, build a model with `build_qlora_model(r=r, alpha=2 * r)`
2.   Train it with `train_adapter(model, steps=QLORA_STEPS, every=1000)` and evaluate it
3.   Append the trainable parameters and the strict accuracy to `rank_results`, then `release("model")` before the next rank

In [ ]:
RANKS = [1, 4, 16, 64]
rank_results = []
for r in RANKS:
    # Your code goes here!
    rank_results.append({"r": r, "trainable": None, "strict": None})

In [ ]:
# Useful cell: accuracy against trainable parameters
ranks_df = pd.DataFrame(rank_results)
plt.semilogx(ranks_df.trainable, ranks_df.strict, marker="o")
for _, row in ranks_df.iterrows():
    plt.annotate(f"r={row.r}", (row.trainable, row.strict))
plt.xlabel("trainable parameters")
plt.ylabel("strict accuracy")
plt.show()

#### **Third coding exercise**: how lossy is NF4?
Compare a de-quantized layer with its original weights:

1.   Load a 4-bit model with `quant_type` and take `model.model.layers[0].self_attn.q_proj.weight`
2.   De-quantize it with `bnb.functional.dequantize_4bit(weight.data, weight.quant_state)`
3.   Return the relative error `‖W_q − W‖ / ‖W‖` against `reference` (in float32)
4.   Compare `"nf4"` and `"fp4"`. Which one fits the weight distribution better?

In [ ]:
# Useful cell: the original weights of the same layer
reference_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32, device_map=device)
reference = reference_model.model.layers[0].self_attn.q_proj.weight.detach().clone()
release("reference_model")
plt.hist(reference.flatten().cpu().numpy(), bins=200)
plt.title("weight distribution of q_proj (layer 0)")
plt.show()

In [ ]:
def quantization_error(quant_type):
    # Your code goes here!
    return None


for quant_type in ["nf4", "fp4"]:
    print(quant_type, quantization_error(quant_type))

#### Last coding exercise: merging the adapter
For serving, an adapter can be folded into the weights: `W ← W + (α/r)·B·A`.

1.   Load the base model in `AMP_DTYPE` (**not** quantized)
2.   Attach the saved adapter with `PeftModel.from_pretrained(base, QLORA_ADAPTER_DIR)` and evaluate it
3.   Call `merge_and_unload()` and evaluate again; the check cell compares the predictions
4.   Why can we not merge into the 4-bit weights directly? And why may the accuracy differ slightly from the QLoRA evaluation above?

In [ ]:
from peft import PeftModel

adapter_results = None
merged_results = None
# Your code goes here!

In [ ]:
# Useful cell: do the merged and unmerged models agree?
agreement = (adapter_results.prediction == merged_results.prediction).mean()
print(f"Identical predictions: {agreement:.1%}")

<a id='section06'></a>
### Attention faster than O(N^2)?: Flash Attention 2

Standard attention computes `softmax(Q·Kᵀ/√d)·V` by writing the full `N×N` score matrix to GPU main memory (HBM), reading it back for the softmax, writing the probabilities and reading them again to multiply by `V`. Moving that matrix around costs more time than the arithmetic: attention is **memory-bound**.

**FlashAttention** never materializes the matrix:
 - **Tiling**: `Q`, `K` and `V` are processed in blocks that fit in the small, fast on-chip SRAM.
 - **Online softmax**: a running maximum and running sum let us combine the softmax of each block without seeing the whole row.
 - **Recomputation**: the backward pass recomputes the blocks instead of storing them.

**FlashAttention 2** reorganizes the work (parallelism over the sequence, a better split between warps, fewer non-matmul operations) and roughly doubles the speed of the first version.

---
***NOTE***
- *Despite the section title, FlashAttention does **not** change the asymptotic cost: it computes exact attention with the same `O(N²·d)` FLOPs.*
- *What changes is **memory**, from `O(N²)` to `O(N)`, and **memory traffic**, from `Θ(N·d + N²)` to `Θ(N²·d²/M)` HBM accesses (`M` = SRAM size). Fewer bytes moved means a large wall-clock speedup.*
- *Sub-quadratic attention exists (sparse, linear and low-rank approximations), but it is **approximate**. FlashAttention is exact.*
- Flash-Attention now has up to v4 version.
- Flash-Attention 2 does not work in Google Collab. You need a GPU with compute capabilities >= 8. This section will otherwise test all other available attention implementations.
---

#### Further Reading
- [FlashAttention](https://arxiv.org/abs/2205.14135)
- [FlashAttention-2](https://arxiv.org/abs/2307.08691)
- [Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867)
- [PyTorch scaled_dot_product_attention](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)

In [ ]:
# Releasing the models of previous sections (if any)
release("model")

In [ ]:
# @title Customize your key variables here
N_HEADS = 8 # @param {type:"integer"}
HEAD_DIM = 64 # @param {type:"integer"}
BLOCK_SIZE = 128 # @param {type:"integer"}
SEQ_LENGTHS = [512, 1024, 2048, 4096, 8192, 16384]

In [ ]:
# Standard attention: builds the full (N x N) score matrix
def naive_attention(q, k, v):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    weights = torch.softmax(scores, dim=-1)
    return weights @ v


# FlashAttention's forward pass in plain PyTorch: only (block x block) scores exist at any time
def tiled_attention(q, k, v, block_size=BLOCK_SIZE):
    scale = 1 / math.sqrt(q.shape[-1])
    output = torch.empty_like(q)
    for i in range(0, q.shape[-2], block_size):
        q_block = q[..., i:i + block_size, :] * scale
        running_max = torch.full(q_block.shape[:-1], float("-inf"), device=q.device, dtype=q.dtype)
        running_sum = torch.zeros(q_block.shape[:-1], device=q.device, dtype=q.dtype)
        accumulator = torch.zeros_like(q_block)
        for j in range(0, k.shape[-2], block_size):
            scores = q_block @ k[..., j:j + block_size, :].transpose(-2, -1)
            new_max = torch.maximum(running_max, scores.amax(dim=-1))
            correction = torch.exp(running_max - new_max)          # rescales what we accumulated so far
            probabilities = torch.exp(scores - new_max.unsqueeze(-1))
            running_sum = running_sum * correction + probabilities.sum(dim=-1)
            accumulator = accumulator * correction.unsqueeze(-1) + probabilities @ v[..., j:j + block_size, :]
            running_max = new_max
        output[..., i:i + block_size, :] = accumulator / running_sum.unsqueeze(-1)
    return output


q, k, v = (torch.randn(1, N_HEADS, 1024, HEAD_DIM, device=device) for _ in range(3))
print("Max abs difference:", (naive_attention(q, k, v) - tiled_attention(q, k, v)).abs().max().item())

#### Benchmarking the kernels
Our tiled version is a teaching tool: written in Python it is slow, because the speed comes from doing all of this inside **one fused GPU kernel**. PyTorch ships such kernels behind `F.scaled_dot_product_attention`, and we can force each backend:
 - `MATH`: the naive algorithm, in C++.
 - `EFFICIENT_ATTENTION`: memory-efficient attention (xformers), works on older GPUs too.
 - `FLASH_ATTENTION`: the FlashAttention 2 kernel. Needs compute capability ≥ 8.0.

In [ ]:
# Benchmarking naive attention against the fused kernels behind torch's scaled_dot_product_attention
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend


def sdpa_with(backend):
    def attention(q, k, v):
        with sdpa_kernel(backend):
            return F.scaled_dot_product_attention(q, k, v)
    return attention


IMPLEMENTATIONS = {
    "naive": naive_attention,
    "sdpa-math": sdpa_with(SDPBackend.MATH),
    "sdpa-mem-efficient": sdpa_with(SDPBackend.EFFICIENT_ATTENTION),
    "sdpa-flash": sdpa_with(SDPBackend.FLASH_ATTENTION),
}


def benchmark_attention(fn, seq_len, repeats=10):
    q, k, v = (torch.randn(1, N_HEADS, seq_len, HEAD_DIM, device=device, dtype=AMP_DTYPE) for _ in range(3))
    try:
        fn(q, k, v)  # warm-up
        with Measure() as m:
            for _ in range(repeats):
                fn(q, k, v)
        return {"ms": 1000 * m.seconds / repeats, "peak GB": m.peak_gb, "status": "ok"}
    except torch.OutOfMemoryError:
        return {"ms": float("nan"), "peak GB": float("nan"), "status": "out of memory"}
    except RuntimeError as error:
        return {"ms": float("nan"), "peak GB": float("nan"), "status": f"unsupported: {str(error)[:60]}"}
    finally:
        del q, k, v
        gc.collect()
        torch.cuda.empty_cache()


rows = []
for name, fn in IMPLEMENTATIONS.items():
    for seq_len in SEQ_LENGTHS:
        rows.append({"implementation": name, "N": seq_len, **benchmark_attention(fn, seq_len)})
attention_results = pd.DataFrame(rows)
print(attention_results[attention_results.status != "ok"].drop_duplicates(["implementation", "status"])
      [["implementation", "N", "status"]].to_string(index=False))
attention_results.pivot(index="N", columns="implementation", values="ms").round(2)

In [ ]:
# Time and peak memory; the dashed line is the size of ONE N x N score matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, group in attention_results.groupby("implementation"):
    axes[0].plot(group.N, group.ms, marker="o", label=name)
    axes[1].plot(group.N, group["peak GB"], marker="o", label=name)
n = np.array(SEQ_LENGTHS)
axes[1].plot(n, N_HEADS * n**2 * torch.finfo(AMP_DTYPE).bits / 8 / 1024**3, "k--", label="one N×N matrix")
for ax, title in zip(axes, ["time (ms)", "peak memory (GB)"]):
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel("sequence length N")
    ax.set_title(title)
    ax.legend()
plt.show()

#### Flash Attention 2 inside transformers
Models choose their attention kernel with `attn_implementation`: `"eager"` (the naive code in Python), `"sdpa"` (PyTorch's fused kernels, the default) or `"flash_attention_2"`. If the `flash-attn` package is not installed, transformers downloads a pre-compiled kernel from the Hub with the `kernels` library, which saves a very long compilation.

We run one forward pass over a long input, computing logits only for the last position (`logits_to_keep=1`) so that the 152k-word vocabulary does not hide the attention cost.

In [ ]:
LONG_CONTEXT = 4096
long_input = torch.randint(0, tokenizer.vocab_size, (1, LONG_CONTEXT), device=device)

candidates = ["eager", "sdpa"] + (["flash_attention_2"] if IS_AMPERE_OR_NEWER else [])
if not IS_AMPERE_OR_NEWER:
    print("Skipping flash_attention_2: it needs compute capability >= 8.0")

rows = []
for implementation in candidates:
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=AMP_DTYPE, device_map=device,
                                                     attn_implementation=implementation)
        with torch.no_grad():
            model(long_input, logits_to_keep=1, use_cache=False)  # warm-up (and kernel download)
            with Measure() as m:
                model(long_input, logits_to_keep=1, use_cache=False)
        rows.append({"requested": implementation, "actually used": model.config._attn_implementation,
                     "ms": 1000 * m.seconds, "peak GB": m.peak_gb})
    except Exception as error:
        print(f"{implementation} not available: {type(error).__name__}: {str(error)[:200]}")
    finally:
        release("model")
pd.DataFrame(rows)

#### **First coding exercise**: causal tiling
Decoder models use **causal** attention: token `i` only sees tokens `≤ i`. Extend the tiled algorithm:

1.   Mask with `-inf` the scores where the key position is greater than the query position
2.   **Skip** the key blocks that are entirely in the future: this is where FlashAttention saves half of the work
3.   Run the check cell, which compares against `F.scaled_dot_product_attention(q, k, v, is_causal=True)`

In [ ]:
def tiled_causal_attention(q, k, v, block_size=BLOCK_SIZE):
    # Your code goes here!
    return None

In [ ]:
# Useful cell: correctness and speed against the non-causal version
q, k, v = (torch.randn(1, N_HEADS, 2048, HEAD_DIM, device=device) for _ in range(3))
reference = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("Max abs difference:", (tiled_causal_attention(q, k, v) - reference).abs().max().item())
for fn in [tiled_attention, tiled_causal_attention]:
    with Measure() as m:
        fn(q, k, v)
    print(f"{fn.__name__:24s} {1000 * m.seconds:.1f} ms")

#### **Second coding exercise**: a memory model
Predict when naive attention runs out of memory:

1.   Write `attention_matrix_bytes(batch, heads, seq_len, dtype)`: the bytes of one `N×N` matrix per head
2.   Naive attention keeps several of these alive at the same time (scores, scaled scores, probabilities). Estimate how many from the `naive` peak memory in `attention_results`
3.   Predict the first `N` that does not fit in `GPU_MEMORY_GB` and check it with `benchmark_attention`

In [ ]:
def attention_matrix_bytes(batch, heads, seq_len, dtype=AMP_DTYPE):
    # Your code goes here!
    return None

In [ ]:
# Useful cell: measured peak vs one matrix
naive = attention_results[(attention_results.implementation == "naive") & (attention_results.status == "ok")]
for _, row in naive.iterrows():
    one = attention_matrix_bytes(1, N_HEADS, row.N) / 1024**3
    print(f"N={row.N:6d}  measured {row['peak GB']:.3f} GB  = {row['peak GB'] / one:.1f} matrices")

#### **Third coding exercise**: exact is not bit-identical
Using `naive_attention` in **float64** as the reference:

1.   Implement `attention_error(fn, seq_len, dtype)`: create float64 `q, k, v`, compute the reference, run `fn` on copies cast to `dtype` and return the max absolute error
2.   Run the useful cell. Which implementation is the most accurate in half precision, and why could a fused kernel beat the naive code?

In [ ]:
def attention_error(fn, seq_len, dtype=AMP_DTYPE):
    # Your code goes here!
    return None

In [ ]:
# Useful cell: error table (NaN where a backend is not supported)
error_rows = []
for name, fn in {**IMPLEMENTATIONS, "tiled": tiled_attention}.items():
    for seq_len in [512, 2048]:
        try:
            error_rows.append({"implementation": name, "N": seq_len, "max error": attention_error(fn, seq_len)})
        except RuntimeError:
            error_rows.append({"implementation": name, "N": seq_len, "max error": float("nan")})
pd.DataFrame(error_rows).pivot(index="implementation", columns="N", values="max error")

#### **Last coding exercise**: time-to-first-token
When a chatbot answers a long document, the first token waits for the whole prompt (the **prefill**).

1.   In `time_to_first_token`, load the model with the given `attn_implementation`, build random `prompt_length` input ids and time `model.generate(input_ids, max_new_tokens=1)` (after one warm-up call)
2.   Release the model before returning
3.   Run the plotting cell for every implementation in `candidates`

In [ ]:
def time_to_first_token(implementation, prompt_length):
    # Your code goes here!
    return None

In [ ]:
# Useful cell: plotting time-to-first-token
PROMPT_LENGTHS = [1024, 2048, 4096, 8192]
for implementation in candidates:
    plt.plot(PROMPT_LENGTHS, [time_to_first_token(implementation, n) for n in PROMPT_LENGTHS],
             marker="o", label=implementation)
plt.xlabel("prompt length")
plt.ylabel("seconds")
plt.legend()
plt.show()

<a id='section07'></a>
### Reinforcement Learning for Language: the TRL library

Supervised fine-tuning teaches a model to **imitate** answers. Often we cannot write the perfect answer, but we can **judge** one. Reinforcement learning lets the model generate, scores the generations with a reward, and pushes up the probability of the better ones.

The usual post-training pipeline is:
1.   **SFT** (Supervised Fine-Tuning) on demonstrations.
2.   **Preference optimization** on pairs of better/worse answers: RLHF with PPO and a learned reward model, or the simpler **DPO**.
3.   **RL with verifiable rewards**: when correctness can be checked by a program (math, code, our dates!), the program *is* the reward.

**GRPO** (Group Relative Policy Optimization) is the algorithm behind step 3 in many recent models:
 - For each prompt, sample a **group** of `G` completions.
 - Score each completion with the reward functions.
 - The **advantage** of a completion is its reward minus the group mean (divided by the group std): no value network is needed.
 - Increase the log-probability of completions with positive advantage and decrease the others, with PPO-style clipping.

[TRL](https://huggingface.co/docs/trl) implements all of these as `Trainer` subclasses (`SFTTrainer`, `DPOTrainer`, `GRPOTrainer`...) that accept a model name, a dataset and a `peft_config`.

---
***NOTE***
- *Generation is the slow part of GRPO. At scale, `GRPOConfig(use_vllm=True)` generates the groups with vLLM, the library of the next section. We keep the plain transformers generation here.*
---

#### Further Reading
- [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155)
- [Direct Preference Optimization](https://arxiv.org/abs/2305.18290)
- [DeepSeekMath (introduces GRPO)](https://arxiv.org/abs/2402.03300)
- [TRL documentation](https://huggingface.co/docs/trl)

In [ ]:
# @title Customize your key variables here
SFT_STEPS = 100 # @param {type:"integer"}
GRPO_STEPS = 80 # @param {type:"integer"}
GRPO_BATCH_SIZE = 16 # @param {type:"integer"}
NUM_GENERATIONS = 8 # @param {type:"integer"}
GRPO_LEARNING_RATE = 5e-5 # @param {type:"number"}
GRPO_ADAPTER_DIR = "adapters/grpo-dates" # @param {type:"string"}

# GRPO_BATCH_SIZE counts completions, so it must be a multiple of NUM_GENERATIONS
assert GRPO_BATCH_SIZE % NUM_GENERATIONS == 0

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer, GRPOConfig, GRPOTrainer

release("model")
precision_flags = {"bf16": AMP_DTYPE == torch.bfloat16, "fp16": AMP_DTYPE == torch.float16}
trl_lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear",
                             task_type="CAUSAL_LM")

#### SFT in a few lines
`SFTTrainer` accepts a dataset with `prompt` and `completion` columns in chat format. It applies the chat template, masks the prompt in the labels and runs the training loop: everything `DateSFTDataset`, `DataCollator` and `train_step` did by hand. Here the base model stays in 16 bits.

In [ ]:
sft_dataset = HFDataset.from_list([
    {"prompt": build_messages(x["text"]), "completion": [{"role": "assistant", "content": x["answer"]}]}
    for x in train_data
])

sft_config = SFTConfig(
    output_dir="outputs/sft-dates",
    max_steps=SFT_STEPS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    learning_rate=2e-4,
    logging_steps=20,
    report_to="none",
    model_init_kwargs={"dtype": AMP_DTYPE},
    **precision_flags,
)
trainer = SFTTrainer(model=MODEL_NAME,
                     args=sft_config,
                     train_dataset=sft_dataset,
                     peft_config=trl_lora_config)
with Measure() as m:
    trainer.train()
record_cost("SFT with TRL (16-bit base, LoRA)", **{"seconds/step": m.seconds / SFT_STEPS, "peak memory (GB)": m.peak_gb})
sft_results = evaluate(trainer.model, valid_data, name="SFT with TRL")
release("trainer")

#### GRPO with verifiable rewards
Reward functions receive the generated `completions` plus every extra column of the dataset (here `answer`) as keyword arguments, and return one float per completion. We combine:
- `correctness_reward`: 1 for an exact answer, plus 0.5 if the right date appears anywhere.
- `format_reward`: 0.25 if the output is a bare `YYYY-MM-DD` string, whatever the date.

**Always test reward functions on hand-made completions before training: a buggy reward trains a buggy model!**

In [ ]:
def completion_text(completion):
    # Conversational datasets produce [{"role": "assistant", "content": ...}]
    return completion[0]["content"] if isinstance(completion, list) else completion


def correctness_reward(completions, answer, **kwargs):
    rewards = []
    for completion, target in zip(completions, answer):
        result = score(completion_text(completion), target)
        rewards.append(result["strict"] + 0.5 * result["lenient"])
    return rewards


def format_reward(completions, **kwargs):
    return [0.25 if re.fullmatch(r"\d{4}-\d{2}-\d{2}", completion_text(c).strip()) else 0.0
            for c in completions]


grpo_dataset = HFDataset.from_list([{"prompt": build_messages(x["text"]), "answer": x["answer"]}
                                    for x in train_data])

# Sanity check on hand-made completions
toy = [[{"role": "assistant", "content": text}] for text in
       ["2021-03-05", "The date is 2021-03-05.", "2021-05-03", "March 5"]]
toy_answers = ["2021-03-05"] * 4
pd.DataFrame({"completion": [completion_text(t) for t in toy],
              "correctness": correctness_reward(toy, toy_answers),
              "format": format_reward(toy)})

In [ ]:
# A small factory so that the exercises can launch variations in one line
def make_grpo_trainer(reward_funcs, output_dir="outputs/grpo-dates", **overrides):
    config_kwargs = dict(
        output_dir=output_dir,
        max_steps=GRPO_STEPS,
        per_device_train_batch_size=GRPO_BATCH_SIZE,
        num_generations=NUM_GENERATIONS,
        max_completion_length=MAX_NEW_TOKENS,
        learning_rate=GRPO_LEARNING_RATE,
        logging_steps=5,
        report_to="none",
        model_init_kwargs={"dtype": AMP_DTYPE},
        **precision_flags,
    )
    config_kwargs.update(overrides)
    return GRPOTrainer(model=MODEL_NAME, reward_funcs=reward_funcs, args=GRPOConfig(**config_kwargs),
                       train_dataset=grpo_dataset, peft_config=trl_lora_config)


def plot_rewards(trainer, extra=("frac_reward_zero_std",)):
    history = pd.DataFrame(trainer.state.log_history).dropna(subset=["reward"])
    columns = ["reward"] + [c for c in history.columns if c.startswith("rewards/") and c.endswith("/mean")]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    history.plot(x="step", y=columns, ax=axes[0], title="mean reward")
    history.plot(x="step", y=[c for c in extra if c in history], ax=axes[1], title="diagnostics")
    plt.show()
    return history

In [ ]:
grpo_trainer = make_grpo_trainer([correctness_reward, format_reward])
with Measure() as m:
    grpo_trainer.train()
record_cost("GRPO with TRL (16-bit base, LoRA)", **{"seconds/step": m.seconds / GRPO_STEPS, "peak memory (GB)": m.peak_gb})

If the reward curve stays flat, raise `GRPO_LEARNING_RATE` (for instance to `1e-4`); if it rises and then collapses, lower it. Reinforcement learning is much more sensitive to this than SFT.

In [ ]:
history = plot_rewards(grpo_trainer)
grpo_results = evaluate(grpo_trainer.model, valid_data, name="GRPO (RL with verifiable rewards)")
grpo_trainer.save_model(GRPO_ADAPTER_DIR)
release("grpo_trainer")

#### **First coding exercise**: shaping with a brevity reward
1.   Write `brevity_reward`: `0` for completions of at most 10 characters, decreasing linearly to `-1` at 50 characters or more
2.   Test it on `toy` with the useful cell
3.   Train with the three rewards and `reward_weights=[1.0, 1.0, 0.5]`, then compare `completions/mean_length` in the training history with the original run

In [ ]:
def brevity_reward(completions, **kwargs):
    # Your code goes here!
    return None

In [ ]:
# Useful cell: test, then train with the extra reward
print(brevity_reward(toy))
brevity_trainer = make_grpo_trainer([correctness_reward, format_reward, brevity_reward],
                                    output_dir="outputs/grpo-brevity", reward_weights=[1.0, 1.0, 0.5])
brevity_trainer.train()
brevity_history = plot_rewards(brevity_trainer, extra=("completions/mean_length",))
release("brevity_trainer")

#### **Second coding exercise**: reward hacking
Rewards get optimized **literally**.

1.   Write `careless_reward`: 1 for any completion containing four consecutive digits, 0 otherwise
2.   Train with it alone for `GRPO_STEPS` steps using the useful cell
3.   Read the sample outputs: the reward went up, but did the model learn to normalize dates?

In [ ]:
def careless_reward(completions, **kwargs):
    # Your code goes here!
    return None

In [ ]:
# Useful cell: training on the careless reward and reading what the model does
hacked_trainer = make_grpo_trainer([careless_reward], output_dir="outputs/grpo-careless")
hacked_trainer.train()
plot_rewards(hacked_trainer)
hacked_results = evaluate(hacked_trainer.model, valid_data, n=64)
print(hacked_results[["text", "answer", "prediction"]].head(10).to_string())
release("hacked_trainer")

#### **Third coding exercise**: DPO
DPO learns from **pairs** without generating during training. Build a preference dataset:

1.   `make_rejected(example, rng)` returns a plausible mistake: swapped day and month, a wrong year, the right date with extra text...
2.   Each row of `dpo_dataset` has `prompt` (the messages), `chosen` and `rejected` (each a one-message assistant list, like `completion` in SFT)
3.   Run the useful cell and compare with SFT and GRPO in the scoreboard

In [ ]:
def make_rejected(example, rng):
    # Your code goes here!
    return None


rng = random.Random(SEED)
dpo_dataset = None  # HFDataset.from_list([...])

In [ ]:
# Useful cell: training with DPOTrainer
from trl import DPOConfig, DPOTrainer

dpo_config = DPOConfig(output_dir="outputs/dpo-dates", max_steps=SFT_STEPS, per_device_train_batch_size=8,
                       learning_rate=5e-5, logging_steps=20, report_to="none",
                       model_init_kwargs={"dtype": AMP_DTYPE}, **precision_flags)
dpo_trainer = DPOTrainer(model=MODEL_NAME, args=dpo_config, train_dataset=dpo_dataset, peft_config=trl_lora_config)
dpo_trainer.train()
evaluate(dpo_trainer.model, valid_data, name="DPO with synthetic preferences")
release("dpo_trainer")

#### **Last coding exercise**: when does a group teach nothing?
If every completion of a group gets the same reward, all advantages are zero and the group gives no learning signal. The trainer logs this as `frac_reward_zero_std`.

1.   Run GRPO with `num_generations` in `[2, 4, 8]` (keep `GRPO_BATCH_SIZE`, which all of them divide)
2.   Store each `plot_rewards(...)` history in `group_histories`
3.   Run the useful cell and relate `frac_reward_zero_std` to the speed of learning

In [ ]:
group_histories = {}
for num_generations in [2, 4, 8]:
    # Your code goes here!
    group_histories[num_generations] = None

In [ ]:
# Useful cell: comparing group sizes
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for g, h in group_histories.items():
    axes[0].plot(h.step, h.reward, label=f"G={g}")
    axes[1].plot(h.step, h.frac_reward_zero_std, label=f"G={g}")
axes[0].set_title("reward")
axes[1].set_title("frac_reward_zero_std")
for ax in axes:
    ax.legend()
plt.show()

<a id='section08'></a>
### Generating language fast: the vLLM library

Generation is a loop: one forward pass per new token. To avoid recomputing the whole prefix at every step, the keys and values of previous tokens are kept in the **KV cache**. Serving many requests is then mostly a **memory management** problem:
 - **PagedAttention** stores the KV cache in fixed-size blocks, like pages of virtual memory: almost no fragmentation, so many more sequences fit at once.
 - **Continuous batching** inserts new requests into the running batch as soon as others finish, instead of waiting for the slowest sequence of a static batch.
 - **Prefix caching** reuses the KV blocks of shared prefixes (system prompts, few-shot examples).
 - **CUDA graphs** and fused kernels remove the Python overhead of each decoding step.

vLLM reserves a fraction of the GPU memory (`gpu_memory_utilization`) for weights and KV cache when it starts, and compiles CUDA graphs: the first start takes a minute or two.

---
***NOTE***
- *We run vLLM inside the notebook process, after freeing the models of earlier sections. If the engine fails to start for lack of memory, restart the runtime, run sections 1 and 2, and come straight here.*
- *Short answers (our dates) mostly measure prompt processing; long answers measure decoding, where vLLM shines. We benchmark both.*
---

#### Further Reading
- [Efficient Memory Management for LLM Serving with PagedAttention](https://arxiv.org/abs/2309.06180)
- [vLLM documentation](https://docs.vllm.ai)

In [ ]:
# @title Customize your key variables here
VLLM_GPU_MEMORY_UTILIZATION = 0.6 # @param {type:"number"}
VLLM_ENFORCE_EAGER = False # @param {type:"boolean"}
N_BENCH = 512 # @param {type:"integer"}
N_LONG = 64 # @param {type:"integer"}
LONG_TOKENS = 256 # @param {type:"integer"}
QLORA_ADAPTER_DIR = "adapters/qlora-dates" # @param {type:"string"}
GRPO_ADAPTER_DIR = "adapters/grpo-dates" # @param {type:"string"}

In [ ]:
# Releasing everything from the previous sections
release("model", "trainer", "grpo_trainer", "brevity_trainer", "hacked_trainer", "dpo_trainer")

# How big is the KV cache of our model?
config = transformers.AutoConfig.from_pretrained(MODEL_NAME)
head_dim = config.hidden_size // config.num_attention_heads
bytes_per_token = 2 * config.num_hidden_layers * config.num_key_value_heads * head_dim * 2  # K and V, 2 bytes each
print(f"KV cache: {bytes_per_token / 1024:.1f} KB per token")
print(f"1 GB of KV cache holds {1024**3 // bytes_per_token:,} tokens, e.g. {1024**3 // bytes_per_token // 2048} sequences of 2048 tokens")
print(f"Free GPU memory now: {free_gpu_gb():.1f} GB")

In [ ]:
# Two workloads: short answers (dates) and long answers (stories)
short_prompts = [build_messages(x["text"]) for x in build_split(N_BENCH, seed=SEED + 2)]
long_prompts = [[{"role": "user", "content": f"Write a short story that takes place on {x['answer']}."}]
                for x in build_split(N_LONG, seed=SEED + 3)]
WORKLOADS = {"short answers": (short_prompts, MAX_NEW_TOKENS), "long answers": (long_prompts, LONG_TOKENS)}
throughput = []


def add_throughput(engine, workload, seconds, generated_tokens):
    throughput.append({"engine": engine, "workload": workload, "seconds": seconds,
                       "generated tokens": generated_tokens, "tokens/s": generated_tokens / seconds})


# Baseline: batched generation with transformers (after a warm-up batch)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=AMP_DTYPE, device_map=device)
generate(model, short_prompts[:EVAL_BATCH_SIZE])
hf_short_outputs = []
for workload, (prompts, max_tokens) in WORKLOADS.items():
    outputs = []
    with Measure() as m:
        for start in range(0, len(prompts), EVAL_BATCH_SIZE):
            outputs += generate(model, prompts[start:start + EVAL_BATCH_SIZE], max_new_tokens=max_tokens)
    tokens = sum(len(tokenizer(o, add_special_tokens=False)["input_ids"]) for o in outputs)
    add_throughput("transformers", workload, m.seconds, tokens)
    if workload == "short answers":
        hf_short_outputs = outputs
release("model")
pd.DataFrame(throughput)

In [ ]:
# Starting the vLLM engine (1-2 minutes the first time)
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # keep the engine inside this notebook process
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from vllm.sampling_params import StructuredOutputsParams

import sys

# Compatibility with notebooks, dirty patch but works!
@contextlib.contextmanager
def real_stdout():
    notebook_stdout = sys.stdout
    sys.stdout = sys.__stdout__ or open(os.devnull, "w")
    try:
        yield
    finally:
        sys.stdout = notebook_stdout

# LoRA ranks of the adapters on disk; vLLM only accepts some maximum ranks
adapter_ranks = [json.load(open(f"{d}/adapter_config.json"))["r"]
                 for d in [QLORA_ADAPTER_DIR, GRPO_ADAPTER_DIR] if os.path.exists(f"{d}/adapter_config.json")]
max_lora_rank = next(r for r in (8, 16, 32, 64, 128, 256) if r >= max(adapter_ranks + [8]))

# Never ask for more memory than is actually free
utilization = min(VLLM_GPU_MEMORY_UTILIZATION, 0.9 * free_gpu_gb() / GPU_MEMORY_GB)
print(f"gpu_memory_utilization = {utilization:.2f} | max_lora_rank = {max_lora_rank}")

with real_stdout():
  llm = LLM(
      model=MODEL_NAME,
      dtype="bfloat16" if AMP_DTYPE == torch.bfloat16 else "half",
      gpu_memory_utilization=utilization,
      max_model_len=4096,
      enable_lora=True,
      max_lora_rank=max_lora_rank,
      enforce_eager=VLLM_ENFORCE_EAGER,
      seed=SEED,
  )
greedy = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS)

In [ ]:
# Same prompts, same greedy decoding: vLLM schedules all of them at once
vllm_short_outputs = []
for workload, (prompts, max_tokens) in WORKLOADS.items():
    start = time.perf_counter()
    outputs = llm.chat(prompts, SamplingParams(temperature=0.0, max_tokens=max_tokens), use_tqdm=False)
    seconds = time.perf_counter() - start
    add_throughput("vLLM", workload, seconds, sum(len(o.outputs[0].token_ids) for o in outputs))
    if workload == "short answers":
        vllm_short_outputs = [o.outputs[0].text for o in outputs]

throughput_df = pd.DataFrame(throughput)
for _, row in throughput_df.iterrows():
    record_cost(f"Generation: {row.engine}, {row.workload}", **{"tokens/s": row["tokens/s"]})
print(throughput_df.pivot(index="workload", columns="engine", values="tokens/s").round(0))

# Greedy decoding should give the same text, up to tiny numerical differences between kernels
agreement = np.mean([a.strip() == b.strip() for a, b in zip(hf_short_outputs, vllm_short_outputs)])
print(f"Identical greedy outputs between transformers and vLLM: {agreement:.1%}")

#### Serving adapters and constraining the output
vLLM applies LoRA adapters **per request** with `LoRARequest`, so one engine serves the base model and all our fine-tuned variants at the same time. It can also force the output to follow a regular expression, a JSON schema or a grammar with **structured outputs**: invalid tokens are masked at every step.

---
***NOTE***
- *The QLoRA adapter was trained on a 4-bit base and is served here on the 16-bit base, so its accuracy may differ slightly from section 5.*
---

In [ ]:
def vllm_evaluate(data, name=None, n=N_EVAL, sampling_params=greedy, lora_request=None, build=build_messages):
    outputs = llm.chat([build(x["text"]) for x in data[:n]], sampling_params,
                       lora_request=lora_request, use_tqdm=False)
    rows = [{**x, "prediction": o.outputs[0].text, **score(o.outputs[0].text, x["answer"])}
            for x, o in zip(data[:n], outputs)]
    return report(pd.DataFrame(rows), name)


regex_params = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS,
                              structured_outputs=StructuredOutputsParams(regex=r"\d{4}-\d{2}-\d{2}"))

print("Base model:          ", end=" "); vllm_evaluate(valid_data, name="vLLM: base model")
print("Base model + regex:  ", end=" "); vllm_evaluate(valid_data, name="vLLM: base model + regex output",
                                                     sampling_params=regex_params)
for lora_id, (label, path) in enumerate({"QLoRA adapter": QLORA_ADAPTER_DIR,
                                         "GRPO adapter": GRPO_ADAPTER_DIR}.items(), start=1):
    if os.path.exists(f"{path}/adapter_config.json"):
        print(f"{label + ':':21s}", end=" ")
        vllm_evaluate(valid_data, name=f"vLLM: {label}", lora_request=LoRARequest(label, lora_id, path))
    else:
        print(f"{label}: not found at {path} (run its section first)")

#### First coding exercise: throughput vs concurrency
Continuous batching shines when many requests are in flight. With the same engine:

1.   For each `n` in `CONCURRENCY`, send the first `n` long prompts in a single `llm.chat` call (`SamplingParams(temperature=0.0, max_tokens=LONG_TOKENS)`)
2.   Measure the seconds and count the generated tokens (`len(o.outputs[0].token_ids)`)
3.   Append `{"n": n, "tokens/s": ...}` to `concurrency_results` and run the plotting cell. Why does it grow almost linearly at first?

In [ ]:
CONCURRENCY = [1, 4, 16, 64]
concurrency_results = []
for n in CONCURRENCY:
    # Your code goes here!
    pass

In [ ]:
# Useful cell: plotting throughput
pd.DataFrame(concurrency_results).plot(x="n", y="tokens/s", marker="o", logx=True, figsize=(6, 3.5))
plt.xlabel("requests in flight")
plt.show()

#### Second coding exercise: prefix caching
vLLM reuses the KV cache of prefixes it has already seen. We do not need a new engine to observe it:

1.   Build `prefix_prompts`: 64 validation sentences, each with the same 32-shot prefix (`build_messages(text, shots=LONG_SHOTS)`)
2.   Call `llm.reset_prefix_cache()`, then time the batch twice with `timed_chat`: a **cold** and a **warm** run
3.   Explain the difference. What does it mean for a chatbot whose system prompt is 2,000 tokens long?

In [ ]:
# Useful cell: a timing helper
LONG_SHOTS = build_split(32, seed=SEED + 4)


def timed_chat(prompts, sampling_params=greedy):
    start = time.perf_counter()
    llm.chat(prompts, sampling_params, use_tqdm=False)
    return time.perf_counter() - start

In [ ]:
prefix_prompts = None
cold_seconds = None
warm_seconds = None
# Your code goes here!
print(f"cold: {cold_seconds} s | warm: {warm_seconds} s")

#### Third coding exercise: self-consistency, the fast way
`SamplingParams(n=8, temperature=0.7, max_tokens=MAX_NEW_TOKENS)` returns 8 completions per prompt in one call, sharing the prompt's KV cache.

1.   Generate 8 samples for every validation prompt with `llm.chat`
2.   For each output, apply `majority_vote([s.text for s in o.outputs])`
3.   Build a results DataFrame with `prediction`, `answer` and the `score` columns and pass it to `report(..., name="vLLM: self-consistency (8 samples)")`

In [ ]:
voting_params = SamplingParams(n=8, temperature=0.7, max_tokens=MAX_NEW_TOKENS, seed=SEED)
voting_results = None
# Your code goes here!

#### Last coding exercise: structured JSON
Ask the model for `{"date": "YYYY-MM-DD", "original": "<the date as written in the text>"}` and enforce the shape with a JSON schema:

1.   Complete `schema` (both fields required strings; add `"pattern"` to `date`) and write a system prompt that asks for that JSON
2.   Generate with `SamplingParams(temperature=0.0, max_tokens=64, structured_outputs=StructuredOutputsParams(json=schema))`
3.   Run the useful cell: every output must parse, but are the values right?

In [ ]:
schema = {
    "type": "object",
    "properties": {
        # Your code goes here!
    },
    "required": ["date", "original"],
}
JSON_SYSTEM_PROMPT = None
json_outputs = None  # the texts generated by vLLM

In [ ]:
# Useful cell: parsing and scoring the JSON
parsed = [json.loads(text) for text in json_outputs]
date_accuracy = np.mean([p["date"] == x["answer"] for p, x in zip(parsed, valid_data)])
found_in_text = np.mean([p["original"] in x["text"] for p, x in zip(parsed, valid_data)])
print(f"All {len(parsed)} outputs parsed | date correct: {date_accuracy:.1%} | 'original' copied from the text: {found_in_text:.1%}")

<a id='section09'></a>
### Closing remarks and scoreboard

Every approach was scored on the same validation examples. The bars show strict and lenient accuracy with their 95% confidence intervals: **when two intervals overlap a lot, do not claim a winner!**

In [ ]:
# The scoreboard
board = pd.DataFrame(SCOREBOARD)
if board.empty:
    print("Nothing evaluated yet: run some sections first!")
else:
    y = np.arange(len(board))
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(board) + 1.5))
    for offset, metric in [(-0.2, "strict"), (0.2, "lenient")]:
        errors = [board[metric] - board[f"{metric}_low"], board[f"{metric}_high"] - board[metric]]
        ax.barh(y + offset, board[metric], height=0.4, xerr=errors, capsize=3, label=metric)
    ax.set_yticks(y, board.approach)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel("accuracy on the validation examples")
    ax.legend()
    plt.show()
board.round(3) if not board.empty else None

In [ ]:
# What each technique cost (time, memory, throughput)
pd.DataFrame(COSTS).round(3)

We took one small model through the whole lifecycle:

| Technique | What achieves | Where to see it |
|---|---|---|
| Prompting | Behaviour change with zero training | zero-shot vs few-shot rows of the scoreboard |
| Automatic Mixed Precision | Much faster matmuls, smaller activations | `COSTS`: full fine-tuning rows |
| QLoRA | Fine-tuning with 4-bit frozen weights and ~1% trainable parameters | QLoRA rows of both tables |
| Flash Attention 2 | Exact attention with linear memory and far less memory traffic | the attention benchmark plots |
| TRL / GRPO | Learning from a reward instead of from demonstrations | SFT vs GRPO rows of the scoreboard |
| vLLM | High-throughput generation, multi-adapter serving, constrained outputs | `COSTS`: generation rows, vLLM rows of the scoreboard |

Please remember that these techniques compose: in practice a model is fine-tuned with QLoRA in bfloat16 with Flash Attention, aligned with GRPO using vLLM to generate the rollouts, and finally served with vLLM.